# 03 - Embedding & Neo4j Ingestion

**Task 1, Fase 3:** Generate embeddings untuk nodes KG dan load ke Neo4j database.

**Pipeline:** `Deduped Triples → Embed → Load Neo4j → Verify`

⚠️ **Pastikan Neo4j sudah running sebelum menjalankan notebook ini.**

In [4]:
# === Setup ===
import sys
import json
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

print(f'Project root: {PROJECT_ROOT}')
print(f'Neo4j URI: {NEO4J_URI}')
print(f'API Key: {GEMINI_API_KEY[:10]}...' if GEMINI_API_KEY else 'ERROR: No API key!')

Project root: d:\TA
Neo4j URI: bolt://localhost:7687
API Key: AIzaSyB38G...


## Step 1: Generate Embeddings

Generate vector embeddings untuk setiap node menggunakan `gemini-embedding-001`.

In [5]:
from pipeline.transform.embedder import embed_triples_file

DEDUPED_PATH = 'data/deduped/UU_11_2008_triples.json'

# Check if embeddings already exist
embedded_path = 'data/embedded/UU_11_2008_triples.json'
if os.path.exists(embedded_path):
    print(f'Embeddings already exist at {embedded_path}')
    print('Delete the file to re-generate.')
else:
    embedded_path = embed_triples_file(
        input_path=DEDUPED_PATH,
        output_dir='data/embedded',
        api_key=GEMINI_API_KEY,
        model='gemini-embedding-001'
    )

# Load and show stats
with open(embedded_path, 'r', encoding='utf-8') as f:
    embedded_data = json.load(f)

embedded_count = sum(1 for n in embedded_data['nodes'] if any(v != 0.0 for v in n.get('embedding', [])))
dim = len(embedded_data['nodes'][0].get('embedding', [])) if embedded_data['nodes'] else 0

print(f'\n=== Embedding Summary ===')
print(f'Total nodes: {embedded_data["total_nodes"]}')
print(f'Nodes with embeddings: {embedded_count}')
print(f'Embedding dimensions: {dim}')
print(f'Model: {embedded_data.get("embedding_model", "unknown")}')

Embeddings already exist at data/embedded/UU_11_2008_triples.json
Delete the file to re-generate.

=== Embedding Summary ===
Total nodes: 436
Nodes with embeddings: 436
Embedding dimensions: 3072
Model: gemini-embedding-001


## Step 2: Connect to Neo4j & Create Indexes

In [7]:
from pipeline.load.neo4j_loader import Neo4jLoader

loader = Neo4jLoader(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

# Uncomment to clear DB first (careful!)
# loader.clear_database()

loader.create_constraints()
loader.create_indexes()

Connected to Neo4j at bolt://localhost:7687
Constraints created.
  Vector index created.
  Full-text index created.
Indexes created.


## Step 3: Load Nodes & Edges

In [8]:
# Load nodes with embeddings
print(f'Loading {len(embedded_data["nodes"])} nodes...')
loader.load_nodes(embedded_data['nodes'])

# Load edges 
print(f'\nLoading {len(embedded_data["edges"])} edges...')
loader.load_edges(embedded_data['edges'])

Loading 436 nodes...


Loading nodes: 100%|██████████| 436/436 [00:25<00:00, 16.84it/s]



Loading 553 edges...


Loading edges: 100%|██████████| 553/553 [00:04<00:00, 132.11it/s]


## Step 4: Verify & Explore

In [9]:
# Get database stats
stats = loader.get_stats()

print(f'=== Neo4j Database Stats ===')
print(f'Total nodes: {stats["total_nodes"]}')
print(f'Total edges: {stats["total_edges"]}')
print(f'\nNode labels:')
for label, count in sorted(stats['node_labels'].items(), key=lambda x: -x[1]):
    print(f'  {label:20s}: {count}')
print(f'\nRelationship types:')
for rel, count in sorted(stats['relationship_types'].items(), key=lambda x: -x[1]):
    print(f'  {rel:20s}: {count}')

=== Neo4j Database Stats ===
Total nodes: 435
Total edges: 553

Node labels:
  KonsepHukum         : 120
  PerbuatanHukum      : 119
  EntitasHukum        : 81
  Pasal               : 74
  Sanksi              : 17
  Bab                 : 14
  UndangUndang        : 10

Relationship types:
  BERLAKU_UNTUK       : 156
  MENGATUR            : 133
  MERUJUK             : 101
  MEMUAT              : 80
  MENDEFINISIKAN      : 64
  MENETAPKAN_SANKSI   : 19


In [10]:
# Test semantic search
query = "pencemaran nama baik"
print(f'=== Semantic Search: "{query}" ===')

try:
    results = loader.test_vector_search(query, GEMINI_API_KEY, top_k=5)
    for r in results:
        print(f'  [{r["type"]}] {r["label"]} (score: {r["score"]:.4f})')
except Exception as e:
    print(f'Vector search not available yet: {e}')
    print('You may need to wait for the vector index to build.')

=== Semantic Search: "pencemaran nama baik" ===
  [PerbuatanHukum] mendistribusikan/mentransmisikan/mengakses konten penghinaan/pencemaran nama baik (score: 0.8893)
  [PerbuatanHukum] melanggar nama Orang terkenal (score: 0.8614)
  [KonsepHukum] melanggar hak Orang lain (score: 0.8568)
  [PerbuatanHukum] penggunaan Nama Domain secara tanpa hak (score: 0.8563)
  [PerbuatanHukum] mengajukan gugatan pembatalan Nama Domain (score: 0.8509)


In [11]:
# Test a simple Cypher query
with loader.driver.session() as session:
    # Find all Pasal and what they regulate
    result = session.run("""
        MATCH (p:Pasal)-[r:MENGATUR]->(ph:PerbuatanHukum)
        RETURN p.label AS pasal, ph.label AS perbuatan
        LIMIT 10
    """)
    print('=== Sample: Pasal → MENGATUR → PerbuatanHukum ===')
    for r in result:
        print(f'  {r["pasal"]} → {r["perbuatan"]}')

=== Sample: Pasal → MENGATUR → PerbuatanHukum ===
  Pasal 20 → Persetujuan atas penawaran Transaksi Elektronik
  Pasal 2 → melakukan perbuatan hukum
  Pasal 3 → Pemanfaatan Teknologi Informasi dan Transaksi Elektronik
  Pasal 4 → Pemanfaatan Teknologi Informasi dan Transaksi Elektronik
  Pasal 5 → penggunaan Sistem Elektronik
  Pasal 7 → memastikan Informasi Elektronik berasal dari Sistem Elektronik yang memenuhi syarat
  Pasal 9 → menyediakan informasi yang lengkap dan benar berkaitan dengan syarat kontrak, produsen, dan produk yang ditawarkan melalui Sistem Elektronik
  Pasal 10 → sertifikasi Transaksi Elektronik
  Pasal 10 → pembentukan Lembaga Sertifikasi Keandalan
  Pasal 10 → menyelenggarakan Transaksi Elektronik


In [12]:
# Clean up
loader.close()
print('Neo4j connection closed.')

Neo4j connection closed.


## Summary

Fase 3 selesai! Knowledge Graph sudah terload di Neo4j.

Output:
- `data/embedded/UU_11_2008_triples.json` — Triples + embeddings
- Neo4j database terisi dengan nodes, edges, vector index, dan full-text index

**Bisa diakses di:** http://localhost:7474 (Neo4j Browser)

**Next:** Task 2 — Fine-tuning LLM untuk Query Generation (NL → Cypher)